In [0]:

CREATE OR REPLACE TABLE workspace.default.gold_fraud_summary AS
SELECT
    step AS hour_bucket,
    type AS transaction_type,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(
        SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 4
    ) AS fraud_rate_pct,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(SUM(CASE WHEN isFraud = 1 THEN amount ELSE 0 END), 2) AS total_fraud_amount
FROM workspace.default.feature_transactions
GROUP BY step, type
ORDER BY step;

In [0]:

SELECT * FROM workspace.default.gold_fraud_summary
ORDER BY fraud_count DESC
LIMIT 10;

In [0]:
SELECT COUNT(*) AS negative_amount_count
FROM workspace.default.feature_transactions
WHERE amount < 0;

In [0]:

SELECT isFraud, COUNT(*) AS row_count
FROM workspace.default.feature_transactions
GROUP BY isFraud;

In [0]:
SELECT nameOrig, nameDest, amount, step, COUNT(*) AS dup_count
FROM workspace.default.feature_transactions
GROUP BY nameOrig, nameDest, amount, step
HAVING COUNT(*) > 1
LIMIT 20;

In [0]:
SELECT
    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS null_amount,
    SUM(CASE WHEN nameOrig IS NULL THEN 1 ELSE 0 END) AS null_orig,
    SUM(CASE WHEN isFraud IS NULL THEN 1 ELSE 0 END) AS null_fraud
FROM workspace.default.feature_transactions;

In [0]:
CREATE OR REPLACE VIEW workspace.default.gold_audit_trail AS
SELECT
    ROW_NUMBER() OVER (ORDER BY step) AS transaction_id,
    event_time,
    type AS transaction_type,
    amount,
    nameOrig AS sender_account,
    nameDest AS receiver_account,
    isFraud AS ground_truth_fraud,
    orig_drained_to_zero AS flag_account_drained,
    CASE
        WHEN isFraud = 1 THEN 'CONFIRMED_FRAUD'
        WHEN orig_drained_to_zero = 1 AND amount > 10000 THEN 'HIGH_RISK_REVIEW'
        ELSE 'NORMAL'
    END AS risk_category
FROM workspace.default.feature_transactions;

In [0]:
SELECT risk_category, COUNT(*) AS count
FROM workspace.default.gold_audit_trail
GROUP BY risk_category
ORDER BY count DESC;